In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
from mxcensus import load_mg_census, load_extended_viviendas
from IPython.display import display, Markdown, Math, HTML
import numpy as np
from scipy.optimize import minimize
from tqdm.auto import tqdm

outputs_path = Path("..") / "outputs"

# Census controls for EOD AGEBs

We take the AGEB identifiers present in the EOD dwelling records and load the corresponding 2020 Census AGEB data. `total_viviendas` is the total count of inhabited private dwellings; `viviendas_caracteristicas` is the count used as the base for the reported dwelling characteristics. We also load bedroom, room, vehicle, bicycle, and internet counts. These Census columns supply candidate controls and dwelling totals, not observed dwelling-type counts for each AGEB. The following display counts all loaded columns, including the AGEB identifier; the eight controls used for calibration are listed below.

In [2]:
# AGEBS from dwelling OD survey data from previous notebook
agebs_from_eod = pd.read_csv(outputs_path / "eod_probabilidades_individuales.csv", usecols=["ageb"], dtype={"ageb": "string"})["ageb"].dropna().drop_duplicates().sort_values().reset_index(drop=True)

In [3]:
# Load census 2020 inhabited private dwellings
census_zones, _ = load_mg_census(state=14)

census_columns = {
    "CVEGEO": "ageb",
    "TVIVPARHAB": "total_viviendas",
    "VIVPARH_CV": "viviendas_caracteristicas",
    "VPH_1DOR": "viviendas_un_dormitorio",
    "VPH_2YMASD": "viviendas_dos_mas_dormitorios",
    "VPH_1CUART": "viviendas_un_cuarto",
    "VPH_2CUART": "viviendas_dos_cuartos",
    "VPH_3YMASC": "viviendas_tres_mas_cuartos",
    "VPH_AUTOM": "acceso_auto_camioneta",
    "VPH_MOTO": "viviendas_moto",
    "VPH_BICI": "viviendas_bici",
    "VPH_INTER": "viviendas_internet"
}
agebs_from_census = census_zones.loc[:, census_columns.keys()].rename(columns=census_columns).copy()
agebs_from_census.reset_index(drop=True, inplace=True)

display(Markdown(f"There are **{len(agebs_from_census['ageb'].unique().tolist()):,}** total AGEBs, which **{agebs_from_census['ageb'].duplicated().sum()}** of them are repeated, "
                 f" and **{agebs_from_census['total_viviendas'].isnull().sum()}** of them have no dwellings reported."
                 f" We will consider a total of **{len(agebs_from_census.columns)}** control variables for the spatial calibration."))

There are **5,873** total AGEBs, which **0** of them are repeated,  and **245** of them have no dwellings reported. We will consider a total of **12** control variables for the spatial calibration.

We retain Census rows for AGEBs present in the EOD and extract the municipality code from each AGEB identifier. The municipality links each AGEB to the donor pool used below.

In [4]:
# Keep both columns of Census 2020
# but keep only the observations from the OD's AGEBS
dwellings_per_ageb = agebs_from_census.loc[agebs_from_census["ageb"].isin(agebs_from_eod)].copy()

display(Markdown(f"From the **{len(agebs_from_census['ageb'].unique().tolist()):,}** total AGEBs we had initially,"
                 f" we will only keep **{len(dwellings_per_ageb['ageb'].unique().tolist()):,}** of them."
                 f" There are  **{dwellings_per_ageb['ageb'].duplicated().sum()}** AGEBs duplicated, **{dwellings_per_ageb['ageb'].isnull().sum()}** AGEB empy values, and **{dwellings_per_ageb['total_viviendas'].isnull().sum()}** values for missing total housing."))

From the **5,873** total AGEBs we had initially, we will only keep **1,046** of them. There are  **0** AGEBs duplicated, **0** AGEB empy values, and **0** values for missing total housing.

In [5]:
# Extract municipality number from AGEB id
dwellings_per_ageb.insert(dwellings_per_ageb.columns.get_loc("ageb") + 1, "municipio", dwellings_per_ageb["ageb"].str[2:5].astype("int64"))

# Extended Survey donors

We load dwelling records from the 2020 Census Extended Survey for Jalisco. Each record supplies a dwelling type, an initial expansion factor, a municipality, and characteristics comparable with the AGEB controls. We keep the nine municipalities selected in the code, so each AGEB can draw on Extended Survey records from its municipality.

In [6]:
dwellings_extended_survey = load_extended_viviendas(state=14)

In [7]:
microdata_columns = {
    "CLAVIVP": "tipo_vivienda_extendido",
    "FACTOR": "factor_expansion",
    "MUN": "municipio",
    "CUADORM": "total_dormitorios",
    "TOTCUART": "total_cuartos", 
    "MOTOCICLETA": "acceso_moto",
    "AUTOPROP": "acceso_auto",
    "BICICLETA": "acceso_bici",
    "INTERNET": "acceso_internet",
}

dwellings_donors_data = dwellings_extended_survey.loc[:, microdata_columns.keys()].rename(columns=microdata_columns).rename_axis("id").copy()

In [8]:
# Keep only the data from ZMG
municipalities_of_interest = [ 39,  44,  51,  70,  97,  98, 101, 120, 124]
dwellings_donors_data = dwellings_donors_data.loc[dwellings_donors_data["municipio"].isin(municipalities_of_interest)]

We standardize the survey labels and convert bedrooms, rooms, and access indicators to numeric values. Unspecified responses become missing values; the indicator functions below treat them as false. We then group the detailed survey dwelling types into four categories shared with the EOD model: `casa_unica`, `casa_compartida_duplex`, `departamento`, and `otra_vivienda`.

In [9]:
# Transform category columns to strings
category_columns = dwellings_donors_data.select_dtypes(include="category").columns
dwellings_donors_data[category_columns] = dwellings_donors_data[category_columns].astype("str")

# Strings with lower case, without spaces, without parenthesis and without accents 
for column in dwellings_donors_data.select_dtypes(include="str").columns:
    dwellings_donors_data[column] = dwellings_donors_data[column].str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii").str.lower().str.replace(r"\s+", "_", regex=True).str.replace("(", "_", regex=False).str.replace(")", "", regex=False)

# Transform not specified data into nan
dwellings_donors_data["total_dormitorios"] = pd.to_numeric(dwellings_donors_data["total_dormitorios"].replace({"-1": pd.NA, "no_especificado": pd.NA})).astype("Int64")
dwellings_donors_data["total_cuartos"] = pd.to_numeric(dwellings_donors_data["total_cuartos"].replace({"-1": pd.NA, "no_especificado": pd.NA})).astype("Int64")
dwellings_donors_data["acceso_moto"] = pd.to_numeric(dwellings_donors_data["acceso_moto"].replace({"si": 1, "no": 0, "blanco_por_pase": pd.NA, "no_especificado": pd.NA})).astype("Int64")
dwellings_donors_data["acceso_auto"] = pd.to_numeric(dwellings_donors_data["acceso_auto"].replace({"si": 1, "no": 0, "blanco_por_pase": pd.NA, "no_especificado": pd.NA})).astype("Int64")
dwellings_donors_data["acceso_bici"] = pd.to_numeric(dwellings_donors_data["acceso_bici"].replace({"si": 1, "no": 0, "blanco_por_pase": pd.NA, "no_especificado": pd.NA})).astype("Int64")
dwellings_donors_data["acceso_internet"] = pd.to_numeric(dwellings_donors_data["acceso_internet"].replace({"si": 1, "no": 0, "blanco_por_pase": pd.NA, "no_especificado": pd.NA})).astype("Int64")

In [10]:
# Generate the dwelling type mapping from the Machine Learning model 
dwelling_type_mapping = {
    "casa_unica_en_el_terreno": "casa_unica",
    "casa_que_comparte_terreno_con_otra_s": "casa_compartida_duplex",
    "casa_duplex": "casa_compartida_duplex",
    "departamento_en_edificio": "departamento",
    "vivienda_en_vecindad_o_cuarteria": "otra_vivienda",
    "vivienda_en_cuarto_de_azotea_de_un_edificio": "otra_vivienda",
    "local_no_construido_para_habitacion": "otra_vivienda",
    "vivienda_movil": "otra_vivienda",
    "refugio": "otra_vivienda",
    "no_especificado_de_vivienda_particular": pd.NA
}

dwellings_donors_data.insert(dwellings_donors_data.columns.get_loc("tipo_vivienda_extendido") + 1, "tipo_vivienda_agrupado", dwellings_donors_data["tipo_vivienda_extendido"].map(dwelling_type_mapping).astype("str"))

# Entropy calibration of donor weights

For each AGEB $g$, we reweight eligible donors $i$ from the same municipality separately. A donor is a survey record used to estimate an AGEB distribution; it need not be a dwelling observed in that AGEB. Let $d_i$ be its original expansion factor, $x_i$ its vector of available control indicators (including the constant for `viviendas_caracteristicas`), and $T_g$ the corresponding Census totals. The calibrated factor is

$$
w^*_{gi}=d_i\exp\!\left(x_i^\top\lambda_g\right),
$$

where $\lambda_g$ is the AGEB-specific vector of calibration parameters. The notebook minimizes the following objective in $\lambda_g$:

$$
F_g(\lambda_g)=\sum_i d_i\exp\!\left(x_i^\top\lambda_g\right)-T_g^\top\lambda_g,
\qquad
\nabla F_g(\lambda_g)=\sum_i w^*_{gi}x_i-T_g.
$$

At a fitted solution, the gradient gives the control constraints

$$
\sum_i w^*_{gi}x_i=T_g.
$$

This objective is the dual form of an entropy-distance adjustment. Its distance interpretation is to choose weights close to $d_i$ by minimizing

$$
\sum_i \left[w_{gi}\log\!\left(\frac{w_{gi}}{d_i}\right)-w_{gi}+d_i\right]
$$

subject to those controls. The code optimizes $F_g$, rather than this distance expression directly; numerical control totals can differ slightly from their targets.

In [11]:
def minimize_distance_entropy(calibration_parameters, donor_attributes_matrix, grantee_totals, initial_expansion_factors):
    calibrated_expansion_factors = initial_expansion_factors * np.exp(donor_attributes_matrix @ calibration_parameters)
    objective = calibrated_expansion_factors.sum() - grantee_totals @ calibration_parameters
    gradient = donor_attributes_matrix.T @ calibrated_expansion_factors - grantee_totals
    return objective, gradient

def reweighting_expansion_factors(donor_dataframe, grantee_dataframe, target_variable, attributes, common_variable, initial_expansion_factors):
    calibrated_donors = []
    valid_donors = donor_dataframe.loc[donor_dataframe[target_variable].notna()].copy()
    donor_groups = valid_donors.groupby(common_variable, sort=False)
    for grantee_index, grantee in tqdm(grantee_dataframe.iterrows(), total=len(grantee_dataframe)):
        donors = donor_groups.get_group(grantee[common_variable]).copy()
        available_attributes = [attribute for attribute in attributes if pd.notna(grantee[attribute])]
        donor_attributes = pd.DataFrame({attribute: attributes[attribute](donors) for attribute in available_attributes}, index=donors.index)
        grantee_totals = grantee[available_attributes].astype(float)
        zero_attributes = [attribute for attribute in available_attributes if grantee_totals[attribute] == 0]
        eligible_donors = ~donor_attributes[zero_attributes].eq(1).any(axis=1)
        positive_attributes = [attribute for attribute in available_attributes if grantee_totals[attribute] > 0]
        expansion_factors_calibrated = pd.Series(0.0, index=donors.index)
        if positive_attributes and eligible_donors.any():
            donor_attributes_matrix = donor_attributes.loc[eligible_donors, positive_attributes].to_numpy(dtype=float)
            positive_totals = grantee_totals.loc[positive_attributes].to_numpy(dtype=float)
            donor_initial_expansion_factors = initial_expansion_factors.loc[donors.index[eligible_donors]].to_numpy(dtype=float)
            result = minimize(minimize_distance_entropy, np.zeros(len(positive_attributes)), args=(donor_attributes_matrix, positive_totals, donor_initial_expansion_factors), jac=True, method="L-BFGS-B")
            expansion_factors_calibrated.loc[eligible_donors] = donor_initial_expansion_factors * np.exp(donor_attributes_matrix @ result.x)
        calibrated_donors.append(pd.DataFrame({"grantee_id": grantee_index, "donor_id": donors.index, common_variable: donors[common_variable].to_numpy(), target_variable: donors[target_variable].to_numpy(), "expansion_factors_calibrated": expansion_factors_calibrated.to_numpy()}))
    return pd.concat(calibrated_donors, ignore_index=True)

## Donors and controls used in the fit

We restrict the calibration pool to the six detailed survey types listed in the following code cell. They map into the four grouped dwelling types; mobile dwellings, shelters, nonresidential premises, and unspecified types are outside this donor pool. For each AGEB, the `calibration_attributes` dictionary uses only these eight controls:

- `viviendas_caracteristicas`: one for every eligible donor, setting the calibrated total for dwellings with characteristics;
- `viviendas_un_dormitorio`: one bedroom;
- `viviendas_un_cuarto` and `viviendas_dos_cuartos`: one or two rooms;
- `acceso_auto_camioneta`, `viviendas_moto`, `viviendas_bici`, and `viviendas_internet`: the corresponding access indicators.

Other Census columns loaded above do not enter the optimization. A missing control is omitted for that AGEB. For a zero-valued control, donors with indicator one receive zero calibrated weight, and the optimizer uses the remaining positive-valued controls and eligible donors.

In [12]:
tipos_con_caracteristicas = ["casa_unica_en_el_terreno", "casa_que_comparte_terreno_con_otra_s", "casa_duplex", "departamento_en_edificio", "vivienda_en_vecindad_o_cuarteria", "vivienda_en_cuarto_de_azotea_de_un_edificio"]
dwellings_donors_calibration = dwellings_donors_data.loc[dwellings_donors_data["tipo_vivienda_extendido"].isin(tipos_con_caracteristicas)].copy()
calibration_attributes = {
    "viviendas_caracteristicas": lambda dataframe: pd.Series(1.0, index=dataframe.index),
    "viviendas_un_dormitorio": lambda dataframe: dataframe["total_dormitorios"].eq(1).fillna(False).astype(float),
    "viviendas_un_cuarto": lambda dataframe: dataframe["total_cuartos"].eq(1).fillna(False).astype(float),
    "viviendas_dos_cuartos": lambda dataframe: dataframe["total_cuartos"].eq(2).fillna(False).astype(float),
    "acceso_auto_camioneta": lambda dataframe: dataframe["acceso_auto"].eq(1).fillna(False).astype(float),
    "viviendas_moto": lambda dataframe: dataframe["acceso_moto"].eq(1).fillna(False).astype(float),
    "viviendas_bici": lambda dataframe: dataframe["acceso_bici"].eq(1).fillna(False).astype(float),
    "viviendas_internet": lambda dataframe: dataframe["acceso_internet"].eq(1).fillna(False).astype(float)
}

calibrated_matrix = reweighting_expansion_factors(
    donor_dataframe=dwellings_donors_calibration,
    grantee_dataframe=dwellings_per_ageb.set_index("ageb", drop=False),
    target_variable="tipo_vivienda_agrupado",
    attributes=calibration_attributes,
    common_variable="municipio",
    initial_expansion_factors=dwellings_donors_data["factor_expansion"]
)

  0%|          | 0/1046 [00:00<?, ?it/s]

## Checking the fitted controls

For one sampled AGEB per municipality, we compare the calibrated donor sum with each available Census control. The relative difference is the weighted sum minus the control, divided by that control; a matched zero control is shown as zero. These tables check the numerical fit of the imposed controls, not the accuracy of the estimated dwelling types.

In [13]:
ageb_controls = dwellings_per_ageb.set_index("ageb")
valid_donors = dwellings_donors_calibration.loc[dwellings_donors_calibration["tipo_vivienda_agrupado"].notna()]
donors_per_municipality = valid_donors.groupby("municipio").size()
calibrated_results = []

for grantee_id, calibrated_ageb in calibrated_matrix.groupby("grantee_id", sort=False):
    municipio = calibrated_ageb["municipio"].iloc[0]
    donors_ageb = dwellings_donors_calibration.loc[calibrated_ageb["donor_id"]]
    available_attributes = [attribute for attribute in calibration_attributes if pd.notna(ageb_controls.loc[grantee_id, attribute])]
    donor_attributes_matrix = pd.DataFrame({attribute: calibration_attributes[attribute](donors_ageb) for attribute in available_attributes}, index=donors_ageb.index)
    weighted_calibrated_sum = donor_attributes_matrix.to_numpy(dtype=float).T @ calibrated_ageb["expansion_factors_calibrated"].to_numpy(dtype=float)
    total_ageb_control = ageb_controls.loc[grantee_id, available_attributes].to_numpy(dtype=float)
    relative_difference = np.full(weighted_calibrated_sum.shape, np.nan, dtype=float)
    np.divide(weighted_calibrated_sum - total_ageb_control, total_ageb_control, out=relative_difference, where=total_ageb_control != 0)
    relative_difference[(total_ageb_control == 0) & np.isclose(weighted_calibrated_sum, 0)] = 0.0
    calibrated_results.append(pd.DataFrame({"grantee_id": grantee_id, "municipio": municipio, "weighted_calibrated_sum": weighted_calibrated_sum, "total_ageb_control": total_ageb_control, "relative_difference": relative_difference, "control": available_attributes, "total_donors": donors_per_municipality.loc[municipio]}))

calibrated_results = pd.concat(calibrated_results, ignore_index=True)

In [14]:
sampled_agebs = calibrated_results[["municipio", "grantee_id"]].drop_duplicates().groupby("municipio", group_keys=False).sample(n=1, random_state=2026)
results_to_display = calibrated_results.merge(sampled_agebs, on=["municipio", "grantee_id"], how="inner")

display(Markdown("## Calibrated results"))
for (municipio, grantee_id), ageb_results in results_to_display.groupby(["municipio", "grantee_id"], sort=False):
    total_donors = ageb_results["total_donors"].iloc[0]
    calibrated_rows = "\n".join(
        f"| {row.weighted_calibrated_sum:,.6f} | {row.total_ageb_control:,.0f} | {row.relative_difference:.6%} | `{row.control}` |"
        for row in ageb_results.itertuples(index=False)
    )
    calibrated_table = f"""### Municipality: {municipio} · AGEB: {grantee_id} · Total donors: {total_donors:,}

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
{calibrated_rows}
"""
    display(Markdown(calibrated_table))

## Calibrated results

### Municipality: 39 · AGEB: 140390001368A · Total donors: 5,733

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 605.009410 | 605 | 0.001555% | `viviendas_caracteristicas` |
| 93.003572 | 93 | 0.003840% | `viviendas_un_dormitorio` |
| 3.997925 | 4 | -0.051871% | `viviendas_dos_cuartos` |
| 423.993944 | 424 | -0.001428% | `acceso_auto_camioneta` |
| 22.997460 | 23 | -0.011043% | `viviendas_moto` |
| 146.004681 | 146 | 0.003206% | `viviendas_bici` |
| 509.013091 | 509 | 0.002572% | `viviendas_internet` |


### Municipality: 44 · AGEB: 1404400540435 · Total donors: 1,858

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 785.998475 | 786 | -0.000194% | `viviendas_caracteristicas` |
| 385.010808 | 385 | 0.002807% | `viviendas_un_dormitorio` |
| 14.001098 | 14 | 0.007841% | `viviendas_un_cuarto` |
| 554.997124 | 555 | -0.000518% | `viviendas_dos_cuartos` |
| 171.000243 | 171 | 0.000142% | `acceso_auto_camioneta` |
| 90.000635 | 90 | 0.000705% | `viviendas_moto` |
| 190.003257 | 190 | 0.001714% | `viviendas_bici` |
| 395.003121 | 395 | 0.000790% | `viviendas_internet` |


### Municipality: 51 · AGEB: 140510016 · Total donors: 2,287

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 852.989881 | 853 | -0.001186% | `viviendas_caracteristicas` |
| 256.001565 | 256 | 0.000611% | `viviendas_un_dormitorio` |
| 21.002751 | 21 | 0.013101% | `viviendas_un_cuarto` |
| 100.996933 | 101 | -0.003037% | `viviendas_dos_cuartos` |
| 605.002982 | 605 | 0.000493% | `acceso_auto_camioneta` |
| 433.003738 | 433 | 0.000863% | `viviendas_moto` |
| 287.997250 | 288 | -0.000955% | `viviendas_bici` |
| 290.000083 | 290 | 0.000028% | `viviendas_internet` |


### Municipality: 70 · AGEB: 1407000130401 · Total donors: 2,740

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 111.996721 | 112 | -0.002928% | `viviendas_caracteristicas` |
| 32.999676 | 33 | -0.000983% | `viviendas_un_dormitorio` |
| 3.999507 | 4 | -0.012318% | `viviendas_un_cuarto` |
| 6.999742 | 7 | -0.003680% | `viviendas_dos_cuartos` |
| 62.997580 | 63 | -0.003841% | `acceso_auto_camioneta` |
| 15.002157 | 15 | 0.014380% | `viviendas_moto` |
| 54.997632 | 55 | -0.004306% | `viviendas_bici` |
| 57.999710 | 58 | -0.000500% | `viviendas_internet` |


### Municipality: 97 · AGEB: 1409708823984 · Total donors: 4,574

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 795.996248 | 796 | -0.000471% | `viviendas_caracteristicas` |
| 239.995223 | 240 | -0.001990% | `viviendas_un_dormitorio` |
| 56.999925 | 57 | -0.000132% | `viviendas_dos_cuartos` |
| 396.982166 | 397 | -0.004492% | `acceso_auto_camioneta` |
| 78.986976 | 79 | -0.016486% | `viviendas_moto` |
| 130.999089 | 131 | -0.000695% | `viviendas_bici` |
| 505.018881 | 505 | 0.003739% | `viviendas_internet` |


### Municipality: 98 · AGEB: 1409800011345 · Total donors: 6,090

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 959.001291 | 959 | 0.000135% | `viviendas_caracteristicas` |
| 235.001478 | 235 | 0.000629% | `viviendas_un_dormitorio` |
| 7.002747 | 7 | 0.039247% | `viviendas_un_cuarto` |
| 94.997187 | 95 | -0.002961% | `viviendas_dos_cuartos` |
| 677.002441 | 677 | 0.000361% | `acceso_auto_camioneta` |
| 90.998548 | 91 | -0.001596% | `viviendas_moto` |
| 221.003315 | 221 | 0.001500% | `viviendas_bici` |
| 769.992558 | 770 | -0.000966% | `viviendas_internet` |


### Municipality: 101 · AGEB: 1410100010452 · Total donors: 6,539

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 1,054.014626 | 1,054 | 0.001388% | `viviendas_caracteristicas` |
| 305.010031 | 305 | 0.003289% | `viviendas_un_dormitorio` |
| 56.005691 | 56 | 0.010162% | `viviendas_un_cuarto` |
| 179.004439 | 179 | 0.002480% | `viviendas_dos_cuartos` |
| 439.994688 | 440 | -0.001207% | `acceso_auto_camioneta` |
| 143.001198 | 143 | 0.000838% | `viviendas_moto` |
| 405.999527 | 406 | -0.000116% | `viviendas_bici` |
| 498.001185 | 498 | 0.000238% | `viviendas_internet` |


### Municipality: 120 · AGEB: 1412000620176 · Total donors: 7,131

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 1,560.001869 | 1,560 | 0.000120% | `viviendas_caracteristicas` |
| 314.999106 | 315 | -0.000284% | `viviendas_un_dormitorio` |
| 10.000251 | 10 | 0.002506% | `viviendas_un_cuarto` |
| 133.000274 | 133 | 0.000206% | `viviendas_dos_cuartos` |
| 505.002367 | 505 | 0.000469% | `acceso_auto_camioneta` |
| 182.001502 | 182 | 0.000825% | `viviendas_moto` |
| 352.001031 | 352 | 0.000293% | `viviendas_bici` |
| 804.998499 | 805 | -0.000187% | `viviendas_internet` |


### Municipality: 124 · AGEB: 141240001018A · Total donors: 2,190

| Weighted calibrated sum | Total AGEB control | Relative difference | Control |
| ----------------------: | -----------------: | ------------------: | :------ |
| 1,081.972507 | 1,082 | -0.002541% | `viviendas_caracteristicas` |
| 327.997049 | 328 | -0.000900% | `viviendas_un_dormitorio` |
| 16.998092 | 17 | -0.011225% | `viviendas_un_cuarto` |
| 118.003775 | 118 | 0.003199% | `viviendas_dos_cuartos` |
| 631.979589 | 632 | -0.003230% | `acceso_auto_camioneta` |
| 457.987937 | 458 | -0.002634% | `viviendas_moto` |
| 231.999782 | 232 | -0.000094% | `viviendas_bici` |
| 599.951098 | 600 | -0.008150% | `viviendas_internet` |


## From donor weights to AGEB dwelling-type totals

We sum calibrated donor weights by grouped dwelling type within each AGEB. These sums estimate type totals among dwellings covered by `viviendas_caracteristicas`. We then add $\max(0,\texttt{total\_viviendas}-\texttt{viviendas\_caracteristicas})$ to `otra_vivienda`. This is a modeling allocation for dwellings outside the characteristics base, not an observed Census count of that dwelling type.

In [15]:
dwelling_types_per_ageb = calibrated_matrix.groupby(["grantee_id", "tipo_vivienda_agrupado"], as_index=False)["expansion_factors_calibrated"].sum()
dwelling_types_per_ageb = dwelling_types_per_ageb.pivot(index="grantee_id", columns="tipo_vivienda_agrupado", values="expansion_factors_calibrated").fillna(0).reset_index()
dwelling_types_per_ageb = dwelling_types_per_ageb.rename(columns={"grantee_id": "ageb", "casa_unica": "total_casas_unicas", "casa_compartida_duplex": "total_casas_compartidas_duplex", "departamento": "total_departamentos", "otra_vivienda": "total_otras_viviendas"})
dwelling_types_per_ageb = dwelling_types_per_ageb.merge(dwellings_per_ageb[["ageb", "total_viviendas", "viviendas_caracteristicas"]], on="ageb", how="left")

dwelling_types_per_ageb["total_estimado_caracteristicas"] = dwelling_types_per_ageb[["total_casas_unicas", "total_casas_compartidas_duplex", "total_departamentos", "total_otras_viviendas"]].sum(axis=1)
dwelling_types_per_ageb["viviendas_sin_caracteristicas"] = (dwelling_types_per_ageb["total_viviendas"] - dwelling_types_per_ageb["viviendas_caracteristicas"]).clip(lower=0)
dwelling_types_per_ageb["total_otras_viviendas"] = dwelling_types_per_ageb["total_otras_viviendas"] + dwelling_types_per_ageb["viviendas_sin_caracteristicas"]

dwelling_types_per_ageb["proporcion_casas_unicas"] = dwelling_types_per_ageb["total_casas_unicas"] / dwelling_types_per_ageb["total_viviendas"]
dwelling_types_per_ageb["proporcion_casas_compartidas_duplex"] = dwelling_types_per_ageb["total_casas_compartidas_duplex"] / dwelling_types_per_ageb["total_viviendas"]
dwelling_types_per_ageb["proporcion_departamentos"] = dwelling_types_per_ageb["total_departamentos"] / dwelling_types_per_ageb["total_viviendas"]
dwelling_types_per_ageb["proporcion_otras_viviendas"] = dwelling_types_per_ageb["total_otras_viviendas"] / dwelling_types_per_ageb["total_viviendas"]

## Proportions and saved estimates

We divide each estimated type total by `total_viviendas` to obtain AGEB dwelling-type proportions. The notebook saves `../outputs/dwelling_types_per_ageb.csv` with the AGEB identifier, both Census dwelling totals, and dictionaries of the four estimated type totals and proportions. These proportions are Census-derived estimates and become the spatial targets in notebook 04.

In [ ]:
dwelling_types_per_ageb["total_vivienda_por_tipo"] = dwelling_types_per_ageb[["total_casas_unicas", "total_casas_compartidas_duplex", "total_departamentos", "total_otras_viviendas"]].rename(columns=lambda column: column.removeprefix("total_")).to_dict("records")
dwelling_types_per_ageb["proporcion_vivienda_por_tipo"] = dwelling_types_per_ageb[["proporcion_casas_unicas", "proporcion_casas_compartidas_duplex", "proporcion_departamentos", "proporcion_otras_viviendas"]].rename(columns=lambda column: column.removeprefix("proporcion_")).to_dict("records")
dwelling_types_per_ageb = dwelling_types_per_ageb[["ageb", "total_viviendas", "viviendas_caracteristicas", "total_vivienda_por_tipo", "proporcion_vivienda_por_tipo"]].copy()
dwelling_types_per_ageb.to_csv(outputs_path / "dwelling_types_per_ageb.csv", index=False)